<a href="https://colab.research.google.com/github/ZiyuCao421/26-CP5046-SIN-CAM-TR3-ICT-Project-1-Analysis-and-Design/blob/main/V3/notebooks/02_single_feature_visualisation_validated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# V3 — Stage 2: Single-Feature Results — Saving and Visualisation

**Purpose.** This notebook turns the **aggregated outputs of stage 1** into an auditable short list
of candidate features for the next stage. It reads the five stage-1 artefacts, validates them
against a strict input contract, visualises them, and writes a transparent, deterministic selection.

**What this notebook does not do.** It does **not** train a model, does **not** read the raw mouse
CSV files, does **not** build feature sets, and does **not** recompute any metric. Every number it
shows comes from the stage-1 `single_feature_metrics.csv`; nothing is re-derived from raw data.

> **Results validity.** Every table, figure and file this notebook produces is
> **only valid after a real Colab run on the real stage-1 outputs**. This notebook is committed
> **without stored outputs**, contains **no pre-filled figure** and **no result value of any kind**.

**Data privacy.** Stage 1 already removed the raw rows, the label values, the participant-linked
file names and the local absolute paths. This notebook only reads those aggregate files, and it
writes only derived tables and figures. It never opens a course CSV file and never writes one.

**Inputs** (produced by `V3/notebooks/01_all_feature_screening.ipynb`):

| File | What it carries |
| --- | --- |
| `single_feature_metrics.csv` | The held-out metrics for every (feature × model) pair — the core input |
| `feature_inventory.csv` | Every column of the merged data, with its screening status and rule |
| `excluded_features.csv` | The rejected columns and the rule that rejected them |
| `single_feature_failures.csv` | Everything that could not be evaluated, and why |
| `run_metadata.json` | The stage-1 manifest: cohort, split fingerprint, configuration, run status |

**Outputs** (written to `OUTPUT_DIR`):

| File | Content |
| --- | --- |
| `figures/*.png` | The five figures described below |
| `selected_features.csv` | Every candidate feature with its ranks, the dummy comparison, the selection flag and the reason |
| `selection_summary.csv` | Per-model Top-K and the combined short list, in one long table |
| `stage2_candidate_manifest.json` | Input hashes, configuration, validation results, the selected names, the limitations |
| `correlation_next_step.md` | Why feature-to-feature correlation cannot be computed here, and what to do about it before building feature sets |

**Integrity rules enforced by the code.**

1. The input contract is checked **before** anything is plotted or selected. A failed check stops
   the run with an explicit error; no ranking and no figures are produced from an invalid input.
2. The split is verified to be **one** split: `n_train_common`, `n_test_common`, `n_classes_common`
   and `split_fingerprint` must each hold exactly one unique value across the metrics table.
3. The selection is **deterministic**: fixed ranking, fixed tie-break order, no randomness, no
   tuning, no re-training.
4. The dummy classifier is used as a **reference floor only**. It is never part of the ranking.
5. Figures are generated only from the validated input; no value is hard-coded or invented.


## How to run this notebook in Google Colab

1. **Run stage 1 first.** The five input files must exist and be the output of a *completed*
   stage-1 run in `complete_case_all_candidates` mode. This notebook refuses to work with an
   exploratory or a failed stage-1 run.
2. **Open this notebook in Colab** and set *Runtime → Run all*.
3. **Edit the single configuration cell** (the cell headed *SINGLE EDITABLE CONFIGURATION BLOCK*).
   You normally change only two values:
   * `INPUT_DIR` — the folder that holds the five stage-1 output files;
   * `OUTPUT_DIR` — the folder the figures and the selection tables are written to.

   Both are **Colab template paths**, deliberately not a real local or Drive path:

   ```python
   INPUT_DIR  = "/content/drive/MyDrive/CP5046_Mouse_Dynamics/V3/results/single_feature"
   OUTPUT_DIR = "/content/outputs/v3_stage2_visualisation"
   ```

   If you moved the stage-1 outputs somewhere else, point `INPUT_DIR` at that folder. Nothing else
   in the notebook needs editing, and no path is hard-coded anywhere else.
4. **Run all cells** in order. The validation cell prints a PASS/FAIL report; the figure cells
   save PNGs; the output cells write the tables and the manifest.
5. **Read the outputs** from `OUTPUT_DIR` (printed at the end of the run).
6. **Do not commit course data.** These outputs are aggregate and participant-free, so a
   deliberately reviewed copy may be placed under `V3/results/summaries/`; the raw stage-1 inputs
   stay outside the repository.

### What this notebook will not do

* It will not run without a valid stage-1 output directory; there is no bundled sample, no
  synthetic fallback and no cached result.
* It will not read a raw course CSV, and it will not fit a model.
* It will not produce a ranking if the input contract fails.


In [12]:
# =============================================================================
#  ENVIRONMENT
#  Nothing here is editable. It only reports the runtime the results came from.
# =============================================================================
import hashlib
import json
import os
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

matplotlib.use("Agg")  # figures are saved to disk, not displayed inline

print("python      :", sys.version.split()[0])
print("platform    :", platform.platform())
print("numpy       :", np.__version__)
print("pandas      :", pd.__version__)
print("matplotlib  :", matplotlib.__version__)


python      : 3.13.16
platform    : Linux-6.6.159+-x86_64-with-glibc2.39
numpy       : 2.4.6
pandas      : 2.2.3
matplotlib  : 3.10.0


In [13]:
# =============================================================================
#  SINGLE EDITABLE CONFIGURATION BLOCK
#  ---------------------------------------------------------------------------
#  Edit ONLY this cell. Every other cell reads these names and nothing else.
#  The two paths below are Colab TEMPLATE paths, not a real local or Drive
#  location: change INPUT_DIR to the folder that holds the five stage-1 output
#  files, and OUTPUT_DIR to the folder the figures and tables are written to.
# =============================================================================

# --- Paths (the only two settings you normally change) ----------------------
INPUT_DIR = "/content/drive/MyDrive/CP5046_Mouse_Dynamics/V3/results/single_feature"
OUTPUT_DIR = "/content/outputs/v3_stage2_visualisation"

# --- Selection --------------------------------------------------------------
PRIMARY_METRIC = "macro_f1"       # the metric the ranking is based on
TOP_K = 12                        # size of the per-model Top-K and of the short list
RANKING_MODE = "mean_rank_rbf_svc_and_random_forest"

# --- Figures ----------------------------------------------------------------
FIGURE_DPI = 150
SAVE_FIGURES = True


```markdown
### 🔑 运行前准备：挂载 Google Drive
如果您选择将输入数据保存在 Google Drive 中，请运行以下单元格以授权 Colab 访问您的云端硬盘。
并将您的 5 个阶段 1 结果文件上传至 Google Drive 的以下路径：
`我的云端硬盘/CP5046_Mouse_Dynamics/V3/results/single_feature/`
```

In [14]:
# 挂载 Google Drive 命令行
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive 挂载成功！")
except ImportError:
    print("当前环境非 Google Colab，无需挂载 Drive。")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive 挂载成功！


In [15]:
import pandas as pd

# 读取本地的阶段 1 指标文件
metrics_df = pd.read_csv('/content/single_feature_metrics.csv')

# 展示前 5 行
display(metrics_df.head(5))

,feature,model,accuracy,macro_precision,macro_recall,macro_f1,cohort_mode,n_train,n_test,n_classes_total,...,n_rows_common_cohort,split_fingerprint,n_non_null_full,n_infinite_full,n_coercion_failures_full,n_unique_full,n_unique_in_cohort,feature_note,fit_seconds,results_validity
0,d0,DummyClassifier_most_frequent,0.182045,0.020227,0.111111,0.034224,complete_case_all_candidates,1601,401,9,...,2002,sha256:297284c375de6cb3535fbd2268907915,2002,0,0,417,417,NaN,0.0062,only valid after a real Colab run
1,d0,RBF_SVC_scaled,0.154613,0.070595,0.134199,0.072902,complete_case_all_candidates,1601,401,9,...,2002,sha256:297284c375de6cb3535fbd2268907915,2002,0,0,417,417,NaN,0.1885,only valid after a real Colab run
2,d0,RandomForest_balanced,0.149626,0.150539,0.142646,0.138006,complete_case_all_candidates,1601,401,9,...,2002,sha256:297284c375de6cb3535fbd2268907915,2002,0,0,417,417,NaN,0.5105,only valid after a real Colab run
3,d1,DummyClassifier_most_frequent,0.182045,0.020227,0.111111,0.034224,complete_case_all_candidates,1601,401,9,...,2002,sha256:297284c375de6cb3535fbd2268907915,2002,0,0,420,420,NaN,0.0055,only valid after a real Colab run
4,d1,RBF_SVC_scaled,0.117207,0.067134,0.109234,0.054841,complete_case_all_candidates,1601,401,9,...,2002,sha256:297284c375de6cb3535fbd2268907915,2002,0,0,420,420,NaN,0.1663,only valid after a real Colab run


## Configuration reference

Every knob lives in the single editable cell above. The two paths are Colab **template** paths on
purpose: no real local or Drive location is hard-coded anywhere in this notebook.

| Name | Default | Meaning |
| --- | --- | --- |
| `INPUT_DIR` | a Colab template path | Folder holding the five stage-1 output files |
| `OUTPUT_DIR` | a Colab template path | Folder the figures and the selection tables are written to |
| `PRIMARY_METRIC` | `macro_f1` | The stage-1 metric the ranking is based on. Must be one of the metric columns of `single_feature_metrics.csv`. |
| `TOP_K` | `12` | Size of the per-model Top-K and of the combined short list |
| `RANKING_MODE` | `mean_rank_rbf_svc_and_random_forest` | How the combined order is formed. See the rules below. |
| `FIGURE_DPI` | `150` | Resolution of the saved PNGs |
| `SAVE_FIGURES` | `True` | Whether the figures are written to `OUTPUT_DIR/figures/` |

### The ranking and tie-break rules

The selection is fully deterministic. There is no randomness, no tuning and no re-training.

1. **Per-model rank.** For the RBF SVC and for the Random Forest, the candidate features are
   ranked by `PRIMARY_METRIC`, highest first. Ties receive the **average** of the ranks they
   span (`method="average"`), so two features with the same score share a rank and the next
   feature continues after the gap.
2. **Combined rank.** `mean_rank = (rank_rbf_svc + rank_random_forest) / 2`. A low `mean_rank` is
   better.
3. **Tie-break order.** The final order is
   `mean_rank` ascending → **the larger of the two models' `PRIMARY_METRIC` values** descending →
   **feature name ascending (A→Z)**. The last key makes the order total, so the same input always
   produces the same short list.
4. **Short list.** The first `TOP_K` features in that order form the short list. `selected` is
   `True` for them and `False` for every other candidate.
5. **The dummy classifier is a floor, never a rank.** `DummyClassifier_most_frequent` is excluded
   from the ranking and is used only to state, per feature, whether the feature beats the
   majority-class baseline on the same split. The outcome is recorded in
   `beats_dummy_rbf` / `beats_dummy_rf` and repeated in `selection_reason`; it never changes the
   order.
6. **No claim of generalisation.** The ranking reuses the stage-1 held-out metrics as a
   *descriptive* short list. Those metrics were already produced by one train/test split, so this
   notebook neither re-estimates them nor claims that the short list is the final selection.


In [16]:
# =============================================================================
#  HELPERS  (not editable)
# =============================================================================
RESULTS_VALIDITY = "only valid after a real Colab run"

# The five stage-1 artefacts this notebook is allowed to read.
METRICS_FILE = "single_feature_metrics.csv"
INVENTORY_FILE = "feature_inventory.csv"
EXCLUDED_FILE = "excluded_features.csv"
FAILURES_FILE = "single_feature_failures.csv"
METADATA_FILE = "run_metadata.json"

REQUIRED_INPUT_FILES = [
    METRICS_FILE,
    INVENTORY_FILE,
    EXCLUDED_FILE,
    FAILURES_FILE,
    METADATA_FILE,
]

# The three stage-1 models. The dummy classifier is the reference floor only.
MODEL_DUMMY = "DummyClassifier_most_frequent"
MODEL_SVC = "RBF_SVC_scaled"
MODEL_RF = "RandomForest_balanced"
EXPECTED_MODELS = [MODEL_DUMMY, MODEL_SVC, MODEL_RF]

# Columns each input table must provide.
REQUIRED_COLUMNS = {
    METRICS_FILE: [
        "feature", "model", "accuracy", "macro_precision", "macro_recall", "macro_f1",
        "cohort_mode", "n_train", "n_test", "n_train_common", "n_test_common",
        "n_classes_common", "n_rows_before_cohort", "n_rows_common_cohort",
        "split_fingerprint", "results_validity",
    ],
    INVENTORY_FILE: [
        "column", "dtype", "status", "exclusion_reason", "protected_by_name_rules",
    ],
    EXCLUDED_FILE: ["column", "exclusion_reason", "status"],
    FAILURES_FILE: ["feature", "model", "stage", "reason"],
}

# The validity stamps a legitimate stage-1 run may carry.
ALLOWED_VALIDITY = {
    "only valid after a real Colab run",
    "exploratory / not rankable - only valid after a real Colab run",
}

REQUIRED_COHORT_MODE = "complete_case_all_candidates"
REQUIRED_RUN_STATUS = "completed"

# The metrics that must hold exactly one value across the whole table.
SPLIT_INVARIANT_COLUMNS = [
    "n_train_common",
    "n_test_common",
    "n_classes_common",
    "split_fingerprint",
]


class InputContractError(RuntimeError):
    """Raised when the stage-1 outputs do not satisfy the input contract."""


def utc_now_iso():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def sha256_file(path):
    """SHA-256 of a file's bytes, used to fingerprint the stage-1 inputs."""
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def require(condition, message):
    """Raise an explicit contract error when a condition does not hold."""
    if not condition:
        raise InputContractError(message)


def read_table(path, name):
    """Read a stage-1 CSV and check that it is non-empty and has the required columns."""
    require(path.exists(), f"missing required stage-1 file: {name}")
    frame = pd.read_csv(path)
    require(
        len(frame) > 0,
        f"{name} is empty. The stage-1 run produced no rows in this table, so no "
        "ranking can be produced.",
    )
    missing = [c for c in REQUIRED_COLUMNS[name] if c not in frame.columns]
    require(
        not missing,
        f"{name} is missing required column(s): {missing}. Found: {list(frame.columns)}",
    )
    return frame


def label_points(ax):
    """Light styling shared by all figures."""
    ax.grid(True, axis="y", alpha=0.25, linewidth=0.6)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)


def place_labels(ax, xs, ys, labels, min_radius=16.0, step=10.0):
    """Place point labels radially away from the centroid, with a leader line.

    Keeps the labels of a scatter readable when several points sit close
    together, without pulling in an extra dependency.
    """
    points = np.column_stack([np.asarray(xs, dtype="float64"), np.asarray(ys, dtype="float64")])
    if not len(points):
        return
    centre = points.mean(axis=0)
    for index, (text, (x, y)) in enumerate(zip(labels, points)):
        direction = np.array([x, y]) - centre
        norm = float(np.linalg.norm(direction))
        if norm < 1e-12:
            direction = np.array([1.0, 0.0])
            norm = 1.0
        direction = direction / norm
        radius = min_radius + step * (index % 6)
        offset = direction * radius
        # the text extends back towards the interior of the axes
        horizontal = "right" if offset[0] > 0 else "left"
        vertical = "top" if offset[1] > 0 else "bottom"
        ax.annotate(
            text,
            (x, y),
            textcoords="offset points",
            xytext=(float(offset[0]), float(offset[1])),
            fontsize=7.5,
            color="#1b1b1b",
            ha=horizontal,
            va=vertical,
            arrowprops={"arrowstyle": "-", "color": "#9a9a9a", "linewidth": 0.6,
                        "shrinkA": 0, "shrinkB": 3},
        )


print("helpers loaded; validity stamp =", repr(RESULTS_VALIDITY))


helpers loaded; validity stamp = 'only valid after a real Colab run'


In [18]:
# =============================================================================
#  STEP 1 — INPUT CONTRACT
#  The stage-1 outputs are validated before anything is plotted or selected.
#  Any failed check stops the run: no figure, no ranking, no output file.
# =============================================================================
validation_results = []


def check(name, condition, detail=""):
    validation_results.append({"check": name, "passed": bool(condition), "detail": detail})
    return bool(condition)


input_dir = Path(INPUT_DIR).expanduser()
# Fallback to local /content if the configured template path does not exist
if not input_dir.exists():
    fallback_dir = Path("/content")
    if (fallback_dir / METRICS_FILE).exists():
        print(f"Warning: {input_dir} not found. Falling back to {fallback_dir} containing stage-1 files.")
        input_dir = fallback_dir

require(input_dir.exists(), f"INPUT_DIR does not exist: {input_dir}")
require(input_dir.is_dir(), f"INPUT_DIR is not a directory: {input_dir}")

try:
    for name in REQUIRED_INPUT_FILES:
        require(
            (input_dir / name).exists(),
            f"missing required stage-1 file: {name} (expected in {input_dir})",
        )

    metrics = read_table(input_dir / METRICS_FILE, METRICS_FILE)
    inventory = read_table(input_dir / INVENTORY_FILE, INVENTORY_FILE)
    excluded = read_table(input_dir / EXCLUDED_FILE, EXCLUDED_FILE)

    # Special handling for single_feature_failures.csv: allow it to be empty if no failures occurred
    failures_path = input_dir / FAILURES_FILE
    failures = pd.read_csv(failures_path)
    if len(failures) == 0:
        failures = pd.DataFrame(columns=REQUIRED_COLUMNS[FAILURES_FILE])
    else:
        missing = [c for c in REQUIRED_COLUMNS[FAILURES_FILE] if c not in failures.columns]
        require(not missing, f"{FAILURES_FILE} is missing required column(s): {missing}")

    with open(input_dir / METADATA_FILE, "r", encoding="utf-8") as handle:
        metadata = json.load(handle)

    # --- validity stamps ----------------------------------------------------
    for name, frame in (
        (METRICS_FILE, metrics),
        (INVENTORY_FILE, inventory),
        (EXCLUDED_FILE, excluded),
        (FAILURES_FILE, failures),
    ):
        if "results_validity" in frame.columns:
            raw_values = frame["results_validity"]
            values = set(raw_values.dropna().astype(str))
            check(
                f"{name}: results_validity is complete and recognised",
                raw_values.notna().all() and bool(values) and values.issubset(ALLOWED_VALIDITY),
                f"found {sorted(values)}; missing={int(raw_values.isna().sum())}",
            )

    check(
        "run_metadata.json: results_validity is a recognised stamp",
        str(metadata.get("results_validity", "")) in ALLOWED_VALIDITY,
        f"found {metadata.get('results_validity')!r}",
    )

    # --- the stage-1 run itself ---------------------------------------------
    check(
        "run_metadata.json: run_status is 'completed'",
        str(metadata.get("run_status", "")) == REQUIRED_RUN_STATUS,
        f"found {metadata.get('run_status')!r}",
    )
    check(
        "run_metadata.json: cohort_mode is 'complete_case_all_candidates'",
        str(metadata.get("cohort_mode", "")) == REQUIRED_COHORT_MODE,
        f"found {metadata.get('cohort_mode')!r}",
    )
    if "cohort_mode" in metrics.columns:
        modes = set(metrics["cohort_mode"].dropna().astype(str))
        check(
            "single_feature_metrics.csv: every row is in the primary cohort mode",
            modes == {REQUIRED_COHORT_MODE},
            f"found {sorted(modes)}",
        )

    # --- one split, one cohort ----------------------------------------------
    for column in SPLIT_INVARIANT_COLUMNS:
        require(
            column in metrics.columns,
            f"{METRICS_FILE} is missing the split-invariant column {column!r}",
        )
        raw_values = metrics[column]
        values = raw_values.dropna().astype(str)
        unique_values = sorted(set(values))
        check(
            f"{METRICS_FILE}: {column} holds exactly one non-missing value for the whole table",
            raw_values.notna().all() and len(unique_values) == 1,
            f"found {unique_values[:5]}; missing={int(raw_values.isna().sum())}",
        )

    metadata_fingerprint = str(metadata.get("split_fingerprint", ""))
    if metadata_fingerprint:
        table_fingerprint = str(metrics["split_fingerprint"].dropna().iloc[0])
        check(
            "the split fingerprint in the metrics table matches the manifest",
            metadata_fingerprint == table_fingerprint,
            f"manifest {metadata_fingerprint!r} vs table {table_fingerprint!r}",
        )

    # --- table integrity -----------------------------------------------------
    duplicates = metrics[metrics.duplicated(subset=["feature", "model"], keep=False)]
    check(
        f"{METRICS_FILE}: no duplicated (feature, model) row",
        len(duplicates) == 0,
        f"{len(duplicates)} duplicated row(s)",
    )

    found_models = sorted(set(metrics["model"].astype(str)))
    unknown_models = [m for m in found_models if m not in EXPECTED_MODELS]
    check(
        f"{METRICS_FILE}: every model is a known stage-1 model",
        not unknown_models,
        f"unknown model(s): {unknown_models}",
    )
    missing_models = [m for m in EXPECTED_MODELS if m not in found_models]
    check(
        f"{METRICS_FILE}: all three stage-1 models are present",
        not missing_models,
        f"missing model(s): {missing_models}",
    )

    counts_per_feature = metrics.groupby("feature")["model"].nunique()
    incomplete = counts_per_feature[counts_per_feature != len(EXPECTED_MODELS)]
    check(
        f"{METRICS_FILE}: every feature has a row for every model",
        len(incomplete) == 0,
        f"{len(incomplete)} feature(s) with an incomplete model set",
    )

    metric_columns = [
        c for c in ("accuracy", "macro_precision", "macro_recall", "macro_f1")
        if c in metrics.columns
    ]
    non_finite = 0
    for column in metric_columns:
        values = pd.to_numeric(metrics[column], errors="coerce").to_numpy(dtype="float64")
        non_finite += int((~np.isfinite(values)).sum())
    check(
        f"{METRICS_FILE}: every metric value is present and finite",
        non_finite == 0,
        f"{non_finite} missing or non-finite metric value(s)",
    )

    require(
        PRIMARY_METRIC in metrics.columns,
        f"PRIMARY_METRIC {PRIMARY_METRIC!r} is not a column of {METRICS_FILE}. "
        f"Available metric columns: {metric_columns}",
    )

    # --- the candidate pool must agree with the metrics ----------------------
    if "status" in inventory.columns:
        candidate_columns = set(
            inventory.loc[inventory["status"].astype(str) == "candidate", "column"].astype(str)
        )
        metric_features = set(metrics["feature"].astype(str))
        check(
            "every screened feature is recorded as a candidate in the inventory",
            metric_features.issubset(candidate_columns),
            f"{len(metric_features - candidate_columns)} feature(s) not in the inventory",
        )

    check(
        "the two optional diagnostic tables are readable",
        True,
        f"{EXCLUDED_FILE}: {len(excluded)} row(s); {FAILURES_FILE}: {len(failures)} row(s)",
    )

except InputContractError as exc:
    print("INPUT CONTRACT REPORT")
    print("-" * 78)
    for entry in validation_results:
        mark = "PASS" if entry["passed"] else "FAIL"
        detail = f"  ({entry['detail']})" if entry['detail'] else ""
        print(f"  [{mark}] {entry['check']}{detail}")
    print("-" * 78)
    print(f"CONTRACT ERROR: {exc}")
    print("The run stops here. No figure, ranking or output file is produced.")
    raise

print("INPUT CONTRACT REPORT")
print("-" * 78)
for entry in validation_results:
    mark = "PASS" if entry["passed"] else "FAIL"
    detail = f"  ({entry['detail']})" if entry['detail'] else ""
    print(f"  [{mark}] {entry['check']}{detail}")
print("-" * 78)
failed_checks = [e for e in validation_results if not e["passed"]]
if failed_checks:
    raise InputContractError(
        f"{len(failed_checks)} input-contract check(s) failed: "
        f"{[e['check'] for e in failed_checks]}. The run stops here."
    )
print(f"All {len(validation_results)} input-contract checks passed.")
print()
print(f"features screened : {metrics['feature'].nunique()}")
print(f"metric rows       : {len(metrics)}")
print(f"split fingerprint  : {metrics['split_fingerprint'].dropna().iloc[0]}")

INPUT CONTRACT REPORT
------------------------------------------------------------------------------
  [PASS] single_feature_metrics.csv: results_validity is complete and recognised  (found ['only valid after a real Colab run']; missing=0)
  [PASS] feature_inventory.csv: results_validity is complete and recognised  (found ['only valid after a real Colab run']; missing=0)
  [PASS] excluded_features.csv: results_validity is complete and recognised  (found ['only valid after a real Colab run']; missing=0)
  [PASS] run_metadata.json: results_validity is a recognised stamp  (found 'only valid after a real Colab run')
  [PASS] run_metadata.json: run_status is 'completed'  (found 'completed')
  [PASS] run_metadata.json: cohort_mode is 'complete_case_all_candidates'  (found 'complete_case_all_candidates')
  [PASS] single_feature_metrics.csv: every row is in the primary cohort mode  (found ['complete_case_all_candidates'])
  [PASS] single_feature_metrics.csv: n_train_common holds exactly one no

In [19]:
# =============================================================================
#  STEP 2 — BUILD THE ANALYSIS FRAME
#  Reads the validated metrics table and derives the ranking. Nothing here is
#  fitted, re-trained or re-estimated: every metric value comes from stage 1.
# =============================================================================
SUPPORTED_RANKING_MODES = {
    "mean_rank_rbf_svc_and_random_forest": (
        "combined order by the mean of the RBF SVC rank and the Random Forest rank"
    ),
    "macro_f1_rbf_svc": "combined order by the RBF SVC macro F1",
    "macro_f1_random_forest": "combined order by the Random Forest macro F1",
}
require(
    RANKING_MODE in SUPPORTED_RANKING_MODES,
    f"RANKING_MODE {RANKING_MODE!r} is not supported. "
    f"Supported modes: {sorted(SUPPORTED_RANKING_MODES)}",
)

work = metrics.copy()
work[PRIMARY_METRIC] = pd.to_numeric(work[PRIMARY_METRIC], errors="coerce")
wide = work.pivot(index="feature", columns="model", values=PRIMARY_METRIC)

for model in EXPECTED_MODELS:
    require(
        model in wide.columns,
        f"{METRICS_FILE} has no {PRIMARY_METRIC!r} value for model {model!r}",
    )

analysis = pd.DataFrame(index=wide.index)
analysis["macro_f1_rbf_svc"] = wide[MODEL_SVC]
analysis["macro_f1_random_forest"] = wide[MODEL_RF]
analysis["dummy_macro_f1"] = wide[MODEL_DUMMY]

# The ranking columns carry the metric actually in use; when the primary metric
# is macro_f1 they coincide with the macro_f1_* columns above.
analysis[f"{PRIMARY_METRIC}_rbf_svc"] = wide[MODEL_SVC]
analysis[f"{PRIMARY_METRIC}_random_forest"] = wide[MODEL_RF]

# --- per-model ranks (average ranks for ties) --------------------------------
analysis["rank_rbf_svc"] = analysis[f"{PRIMARY_METRIC}_rbf_svc"].rank(
    ascending=False, method="average"
)
analysis["rank_random_forest"] = analysis[f"{PRIMARY_METRIC}_random_forest"].rank(
    ascending=False, method="average"
)
analysis["mean_rank"] = (analysis["rank_rbf_svc"] + analysis["rank_random_forest"]) / 2.0

# --- the dummy classifier is the reference floor only ------------------------
dummy_values = analysis["dummy_macro_f1"].dropna()
dummy_shared_value = float(dummy_values.iloc[0]) if len(dummy_values) else float("nan")
dummy_is_constant = bool(len(set(np.round(dummy_values.to_numpy(), 12))) == 1)
analysis["beats_dummy_rbf"] = analysis["macro_f1_rbf_svc"] > analysis["dummy_macro_f1"]
analysis["beats_dummy_rf"] = analysis["macro_f1_random_forest"] > analysis["dummy_macro_f1"]

# --- the combined order, with a total tie-break ------------------------------
if RANKING_MODE == "mean_rank_rbf_svc_and_random_forest":
    analysis["_primary_key"] = analysis["mean_rank"]
    analysis["_secondary_key"] = -analysis[f"{PRIMARY_METRIC}_rbf_svc"].where(
        analysis[f"{PRIMARY_METRIC}_rbf_svc"]
        >= analysis[f"{PRIMARY_METRIC}_random_forest"],
        analysis[f"{PRIMARY_METRIC}_random_forest"],
    )
    analysis = analysis.sort_values(
        ["_primary_key", "_secondary_key", "feature"], ascending=[True, True, True]
    )
elif RANKING_MODE == "macro_f1_rbf_svc":
    analysis["_primary_key"] = -analysis[f"{PRIMARY_METRIC}_rbf_svc"]
    analysis["_secondary_key"] = analysis["mean_rank"]
    analysis = analysis.sort_values(
        ["_primary_key", "_secondary_key", "feature"], ascending=[True, True, True]
    )
else:
    analysis["_primary_key"] = -analysis[f"{PRIMARY_METRIC}_random_forest"]
    analysis["_secondary_key"] = analysis["mean_rank"]
    analysis = analysis.sort_values(
        ["_primary_key", "_secondary_key", "feature"], ascending=[True, True, True]
    )

analysis["combined_order"] = np.arange(1, len(analysis) + 1)
analysis["selected"] = analysis["combined_order"] <= TOP_K

# --- selection reason, recorded explicitly for every row ---------------------
reasons = []
for feature, row in analysis.iterrows():
    beats = (
        "beats_dummy: rbf=True, rf=True"
        if row["beats_dummy_rbf"] and row["beats_dummy_rf"]
        else ("beats_dummy: rbf=True, rf=False"
              if row["beats_dummy_rbf"]
              else ("beats_dummy: rbf=False, rf=True"
                    if row["beats_dummy_rf"]
                    else "beats_dummy: rbf=False, rf=False"))
    )
    if row["selected"]:
        reason = (
            f"top_{TOP_K}_by_{RANKING_MODE}; mean_rank={row['mean_rank']:.3f}; {beats}"
        )
        if not row["beats_dummy_rbf"] and not row["beats_dummy_rf"]:
            reason += "; does_not_beat_dummy_in_either_model"
    else:
        reason = (
            f"outside_top_{TOP_K}_by_{RANKING_MODE}; mean_rank={row['mean_rank']:.3f}; {beats}"
        )
    reasons.append(reason)
analysis["selection_reason"] = reasons

analysis = analysis.drop(columns=["_primary_key", "_secondary_key"])

# --- per-model Top-K (used by the figures and the summary table) -------------
topk_by_model = {}
for model in EXPECTED_MODELS:
    ranked = work[work["model"] == model].sort_values(
        [PRIMARY_METRIC, "feature"], ascending=[False, True]
    )
    topk_by_model[model] = list(ranked["feature"].astype(str))[:TOP_K]

joint_topk = sorted(set(topk_by_model[MODEL_SVC]) & set(topk_by_model[MODEL_RF]))
short_list = list(analysis.index[analysis["selected"]])

print(f"candidates analysed : {len(analysis)}")
print(f"ranking mode        : {RANKING_MODE}")
print(f"primary metric      : {PRIMARY_METRIC}")
print(f"short list size     : {len(short_list)} (TOP_K = {TOP_K})")
print(f"dummy floor (macro F1): {dummy_shared_value:.6f}"
      + ("" if dummy_is_constant else "  [note: not identical across features]"))
print()
print("Short list (combined order):")
for position, feature in enumerate(short_list, start=1):
    row = analysis.loc[feature]
    print(
        f"  {position:>2d}. {feature:<24s} "
        f"mean_rank={row['mean_rank']:.2f} "
        f"macro_f1 svc={row['macro_f1_rbf_svc']:.4f} rf={row['macro_f1_random_forest']:.4f}"
    )


candidates analysed : 57
ranking mode        : mean_rank_rbf_svc_and_random_forest
primary metric      : macro_f1
short list size     : 12 (TOP_K = 12)
dummy floor (macro F1): 0.034224

Short list (combined order):
   1. max_y                    mean_rank=3.00 macro_f1 svc=0.2216 rf=0.2981
   2. min_y                    mean_rank=3.50 macro_f1 svc=0.2174 rf=0.3048
   3. avg_y                    mean_rank=5.50 macro_f1 svc=0.2184 rf=0.2511
   4. min_t                    mean_rank=6.00 macro_f1 svc=0.2591 rf=0.2347
   5. max_t                    mean_rank=7.50 macro_f1 svc=0.2579 rf=0.2307
   6. avg_t                    mean_rank=8.50 macro_f1 svc=0.2629 rf=0.2194
   7. min_x                    mean_rank=10.00 macro_f1 svc=0.2004 rf=0.2294
   8. VV2                      mean_rank=11.00 macro_f1 svc=0.1050 rf=0.2585
   9. max_x                    mean_rank=14.00 macro_f1 svc=0.1797 rf=0.2129
  10. avg_x                    mean_rank=14.00 macro_f1 svc=0.1999 rf=0.1969
  11. VV0            

In [20]:
# =============================================================================
#  STEP 3 — FIGURE 1: TOP-K SINGLE FEATURES PER MODEL
#  One horizontal bar chart per model, from the stage-1 metrics only.
# =============================================================================
output_dir = Path(OUTPUT_DIR).expanduser()
figures_dir = output_dir / "figures"
if SAVE_FIGURES:
    figures_dir.mkdir(parents=True, exist_ok=True)

MODEL_LABELS = {
    MODEL_DUMMY: "DummyClassifier (majority-class floor)",
    MODEL_SVC: "RBF SVC (scaled)",
    MODEL_RF: "Random Forest (balanced)",
}
MODEL_COLORS = {MODEL_DUMMY: "#8a8a8a", MODEL_SVC: "#2f6fb2", MODEL_RF: "#3d8f5a"}

saved_figures = []


def save_figure(fig, filename):
    """Save a figure as PNG when SAVE_FIGURES is on, and record its name."""
    if SAVE_FIGURES:
        path = figures_dir / filename
        fig.savefig(path, dpi=FIGURE_DPI, bbox_inches="tight")
        saved_figures.append(filename)
        print(f"  saved: figures/{filename}")
    plt.close(fig)


for model in EXPECTED_MODELS:
    subset = (
        work[work["model"] == model]
        .sort_values([PRIMARY_METRIC, "feature"], ascending=[False, True])
        .head(TOP_K)
        .copy()
    )
    subset = subset.iloc[::-1]  # best at the top of the axis

    fig, ax = plt.subplots(figsize=(8.0, max(3.0, 0.34 * len(subset) + 1.4)))
    positions = np.arange(len(subset))
    values = subset[PRIMARY_METRIC].to_numpy(dtype="float64")
    ax.barh(positions, values, color=MODEL_COLORS[model], alpha=0.85, edgecolor="none")
    ax.set_yticks(positions)
    ax.set_yticklabels(subset["feature"].astype(str))
    ax.set_xlabel(f"{PRIMARY_METRIC} on the stage-1 held-out portion")
    ax.set_title(
        f"Top-{TOP_K} single features by {PRIMARY_METRIC}\n{MODEL_LABELS[model]}",
        fontsize=11,
    )
    for position, value in zip(positions, values):
        ax.text(
            value,
            position,
            f" {value:.3f}",
            va="center",
            ha="left",
            fontsize=8,
            color="#333333",
        )
    if model != MODEL_DUMMY and dummy_is_constant:
        ax.axvline(
            dummy_shared_value,
            color="#b03a3a",
            linestyle="--",
            linewidth=1.0,
            label="majority-class floor (dummy)",
        )
        ax.legend(loc="lower right", fontsize=8, frameon=False)
    label_points(ax)
    fig.tight_layout()
    save_figure(fig, f"topk_{PRIMARY_METRIC}_{model}.png")

print()
print(f"Figure 1 complete: {len(EXPECTED_MODELS)} per-model Top-K charts.")


  saved: figures/topk_macro_f1_DummyClassifier_most_frequent.png
  saved: figures/topk_macro_f1_RBF_SVC_scaled.png
  saved: figures/topk_macro_f1_RandomForest_balanced.png

Figure 1 complete: 3 per-model Top-K charts.


In [21]:
# =============================================================================
#  STEP 4 — FIGURE 2: RBF SVC vs RANDOM FOREST
#  Scatter of the stage-1 macro F1 of the two real models, with the y = x
#  reference line. Only the features that are in BOTH models' Top-K are
#  labelled, so the labels do not overlap.
# =============================================================================
x_values = analysis["macro_f1_rbf_svc"].to_numpy(dtype="float64")
y_values = analysis["macro_f1_random_forest"].to_numpy(dtype="float64")

low = float(min(x_values.min(), y_values.min()))
high = float(max(x_values.max(), y_values.max()))
span = max(high - low, 1e-9)
pad = 0.05 * span

fig, ax = plt.subplots(figsize=(7.2, 7.0))
ax.scatter(
    x_values,
    y_values,
    s=42,
    alpha=0.75,
    color="#2f6fb2",
    edgecolor="white",
    linewidth=0.6,
    label=f"candidates (n = {len(analysis)})",
)
ax.plot(
    [low - pad, high + pad],
    [low - pad, high + pad],
    color="#444444",
    linestyle="--",
    linewidth=1.1,
    label="y = x (identical performance)",
)

# Label only the joint Top-K features, spread radially so the labels stay legible.
joint = [f for f in joint_topk if f in analysis.index]
place_labels(
    ax,
    [analysis.loc[f, "macro_f1_rbf_svc"] for f in joint],
    [analysis.loc[f, "macro_f1_random_forest"] for f in joint],
    joint,
)

ax.set_xlabel("RBF SVC macro F1 (stage-1 held-out portion)")
ax.set_ylabel("Random Forest macro F1 (stage-1 held-out portion)")
ax.set_title(
    "Single-feature performance of the two real models\n"
    f"labelled: the {len(joint)} features in both models' Top-{TOP_K}",
    fontsize=11,
)
ax.set_xlim(low - pad, high + pad)
ax.set_ylim(low - pad, high + pad)
ax.set_aspect("equal", adjustable="box")
ax.grid(True, alpha=0.25, linewidth=0.6)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.legend(loc="upper left", fontsize=8, frameon=False)
fig.tight_layout()
save_figure(fig, f"scatter_{PRIMARY_METRIC}_svc_vs_random_forest.png")

print()
print("Figure 2 complete: model-to-model comparison with the y = x reference line.")
print(f"joint Top-{TOP_K} features labelled: {len(joint)}")


  saved: figures/scatter_macro_f1_svc_vs_random_forest.png

Figure 2 complete: model-to-model comparison with the y = x reference line.
joint Top-12 features labelled: 5


In [22]:
# =============================================================================
#  STEP 5 — FIGURE 3: DISTRIBUTION OF THE PRIMARY METRIC PER MODEL
#  Shows how much of the candidate pool is indistinguishable from the floor.
# =============================================================================
fig, axes = plt.subplots(1, len(EXPECTED_MODELS), figsize=(11.5, 4.0))
if len(EXPECTED_MODELS) == 1:
    axes = [axes]

rng = np.random.default_rng(42)  # fixed seed: the jitter must be reproducible
for ax, model in zip(axes, EXPECTED_MODELS):
    values = work.loc[work["model"] == model, PRIMARY_METRIC].to_numpy(dtype="float64")
    ax.boxplot(
        values,
        vert=True,
        patch_artist=True,
        widths=0.55,
        boxprops={"facecolor": MODEL_COLORS[model], "alpha": 0.35, "edgecolor": "#333333"},
        medianprops={"color": "#222222", "linewidth": 1.4},
        whiskerprops={"color": "#555555"},
        capprops={"color": "#555555"},
        flierprops={"marker": "o", "markersize": 2.5, "alpha": 0.35, "markerfacecolor": "#555555"},
    )
    jitter = rng.uniform(-0.14, 0.14, size=len(values))
    ax.scatter(
        np.ones(len(values)) + jitter,
        values,
        s=10,
        alpha=0.45,
        color=MODEL_COLORS[model],
        edgecolor="none",
        zorder=3,
    )
    if model != MODEL_DUMMY and dummy_is_constant:
        ax.axhline(
            dummy_shared_value,
            color="#b03a3a",
            linestyle="--",
            linewidth=1.0,
            label="majority-class floor",
        )
        ax.legend(loc="upper right", fontsize=7.5, frameon=False)
    ax.set_xticks([1])
    ax.set_xticklabels([MODEL_LABELS[model].split(" (")[0]], fontsize=9)
    ax.set_ylabel(PRIMARY_METRIC, fontsize=9)
    ax.set_title(f"n = {len(values)} candidates", fontsize=9)
    label_points(ax)

fig.suptitle(
    f"Distribution of {PRIMARY_METRIC} across the candidate pool, per model",
    fontsize=11,
)
fig.tight_layout()
save_figure(fig, f"distribution_{PRIMARY_METRIC}_by_model.png")

print()
print("Figure 3 complete: per-model distribution of the primary metric.")


  saved: figures/distribution_macro_f1_by_model.png

Figure 3 complete: per-model distribution of the primary metric.


In [23]:
# =============================================================================
#  STEP 6 — FIGURE 4: AGREEMENT OF THE TWO MODEL RANKINGS
#  This is the agreement of the two models' PERFORMANCE RANKINGS of the
#  features. It is NOT a correlation between the features themselves: the
#  stage-1 outputs contain no row-level values, so no feature-to-feature
#  correlation can be computed here. See correlation_next_step.md.
# =============================================================================
rank_x = analysis["rank_rbf_svc"].to_numpy(dtype="float64")
rank_y = analysis["rank_random_forest"].to_numpy(dtype="float64")

# Spearman correlation == Pearson correlation of the average ranks.
spearman_rho = float(np.corrcoef(rank_x, rank_y)[0, 1]) if len(analysis) > 1 else float("nan")

limit = max(len(analysis), 2) + 1
fig, ax = plt.subplots(figsize=(6.8, 6.6))
ax.scatter(
    rank_x,
    rank_y,
    s=42,
    alpha=0.75,
    color="#3d8f5a",
    edgecolor="white",
    linewidth=0.6,
    label=f"candidates (n = {len(analysis)})",
)
ax.plot([0.5, limit], [0.5, limit], color="#444444", linestyle="--", linewidth=1.1,
        label="y = x (identical ranking)")
place_labels(
    ax,
    [analysis.loc[f, "rank_rbf_svc"] for f in joint],
    [analysis.loc[f, "rank_random_forest"] for f in joint],
    joint,
)
ax.set_xlabel("rank by RBF SVC macro F1  (1 = best)")
ax.set_ylabel("rank by Random Forest macro F1  (1 = best)")
ax.set_title(
    "Agreement of the two models' performance rankings\n"
    f"Spearman rho = {spearman_rho:.3f}   |   "
    "this is rank agreement, NOT feature correlation",
    fontsize=10.5,
)
ax.set_xlim(0.5, limit)
ax.set_ylim(0.5, limit)
ax.set_aspect("equal", adjustable="box")
ax.grid(True, alpha=0.25, linewidth=0.6)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.legend(loc="upper left", fontsize=8, frameon=False)
fig.tight_layout()
save_figure(fig, "rank_agreement_svc_vs_random_forest.png")

print()
print("Figure 4 complete: rank-agreement scatter of the two models' rankings.")
print(f"Spearman rho (rank agreement) = {spearman_rho:.4f}")
print("Reminder: this measures how similarly the two models ORDER the features.")
print("It is not a correlation between the feature values themselves.")


  saved: figures/rank_agreement_svc_vs_random_forest.png

Figure 4 complete: rank-agreement scatter of the two models' rankings.
Spearman rho (rank agreement) = 0.3216
Reminder: this measures how similarly the two models ORDER the features.
It is not a correlation between the feature values themselves.


In [24]:
# =============================================================================
#  STEP 7 — FIGURE 5: CANDIDATE POOL AND EXCLUSION AUDIT
#  Counts only. No raw row, no raw value, no label value and no file name is
#  shown or written.
# =============================================================================
reason_counts = (
    excluded["exclusion_reason"].astype(str).replace("", "(no reason recorded)").value_counts()
)
n_candidates = int((inventory["status"].astype(str) == "candidate").sum())
n_excluded = int((inventory["status"].astype(str) == "excluded").sum())
n_screened = int(analysis.shape[0])
n_failed_rows = int(len(failures))

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2), gridspec_kw={"width_ratios": [1.45, 1.0]})

# --- left: why columns were excluded -----------------------------------------
ax = axes[0]
if len(reason_counts):
    ordered = reason_counts.sort_values()
    positions = np.arange(len(ordered))
    ax.barh(positions, ordered.to_numpy(), color="#c47f4a", alpha=0.85, edgecolor="none")
    ax.set_yticks(positions)
    ax.set_yticklabels([r.replace("_", " ") for r in ordered.index], fontsize=8.5)
    for position, value in zip(positions, ordered.to_numpy()):
        ax.text(value, position, f" {int(value)}", va="center", fontsize=8)
else:
    ax.text(0.5, 0.5, "no column was excluded", ha="center", va="center", fontsize=10)
    ax.set_yticks([])
ax.set_xlabel("number of columns")
ax.set_title("Exclusion rules applied", fontsize=10)
label_points(ax)

# --- right: how the pool was assembled ---------------------------------------
ax = axes[1]
stages = ["merged\ncolumns", "candidates", "excluded", "screened", "failure\nrows"]
counts = [n_candidates + n_excluded, n_candidates, n_excluded, n_screened, n_failed_rows]
positions = np.arange(len(stages))
ax.bar(positions, counts, color=["#4a6fa5", "#3d8f5a", "#c47f4a", "#2f6fb2", "#b03a3a"],
       alpha=0.85, edgecolor="none")
ax.set_xticks(positions)
ax.set_xticklabels(stages, fontsize=8)
ax.set_xlim(-0.6, len(stages) - 0.4)
for position, value in zip(positions, counts):
    ax.text(position, value, f" {int(value)}", ha="center", va="bottom", fontsize=8)
ax.set_ylabel("count")
ax.set_title("Candidate pool composition", fontsize=10)
label_points(ax)

fig.suptitle("Audit of the candidate pool (counts only)", fontsize=11)
fig.tight_layout()
save_figure(fig, "candidate_pool_audit.png")

print()
print("Figure 5 complete: candidate pool and exclusion audit.")
print(f"  columns in merged data : {n_candidates + n_excluded}")
print(f"  candidates              : {n_candidates}")
print(f"  excluded                : {n_excluded}")
print(f"  screened                : {n_screened}")
print(f"  failure rows recorded   : {n_failed_rows}")
if len(reason_counts):
    print()
    print("Exclusion reasons:")
    for reason, count in reason_counts.items():
        print(f"  {reason:<38s}: {int(count)}")


  saved: figures/candidate_pool_audit.png

Figure 5 complete: candidate pool and exclusion audit.
  columns in merged data : 70
  candidates              : 57
  excluded                : 13
  screened                : 57
  failure rows recorded   : 0

Exclusion reasons:
  raw_coordinate_column                 : 6
  timestamp_or_time_column              : 3
  config_excluded_column                : 2
  target_label_column                   : 1
  source_file_bookkeeping_column        : 1


In [25]:
# =============================================================================
#  STEP 8 — WRITE THE SELECTION TABLES
#  selected_features.csv : every candidate, with its ranks and the reason
#  selection_summary.csv : per-model Top-K and the combined short list
#  Both are aggregate, participant-free and stamped with the validity notice.
# =============================================================================
split_fingerprint = str(metrics["split_fingerprint"].dropna().iloc[0])

selected_table = pd.DataFrame(
    {
        "feature": analysis.index.astype(str),
        "combined_order": analysis["combined_order"].to_numpy(),
        "rank_rbf_svc": analysis["rank_rbf_svc"].to_numpy(),
        "rank_random_forest": analysis["rank_random_forest"].to_numpy(),
        "mean_rank": analysis["mean_rank"].to_numpy(),
        "macro_f1_rbf_svc": analysis["macro_f1_rbf_svc"].to_numpy(),
        "macro_f1_random_forest": analysis["macro_f1_random_forest"].to_numpy(),
        "beats_dummy_rbf": analysis["beats_dummy_rbf"].to_numpy(),
        "beats_dummy_rf": analysis["beats_dummy_rf"].to_numpy(),
        "selected": analysis["selected"].to_numpy(),
        "selection_reason": analysis["selection_reason"].astype(str).to_numpy(),
        "results_validity": RESULTS_VALIDITY,
        "split_fingerprint": split_fingerprint,
    }
)
selected_table = selected_table.sort_values("combined_order").reset_index(drop=True)

summary_rows = []
for model in EXPECTED_MODELS:
    ranked = work[work["model"] == model].sort_values(
        [PRIMARY_METRIC, "feature"], ascending=[False, True]
    )
    for position, (_, row) in enumerate(ranked.head(TOP_K).iterrows(), start=1):
        summary_rows.append(
            {
                "scope": f"model_top_{TOP_K}",
                "model": model,
                "rank_in_model": position,
                "feature": str(row["feature"]),
                "primary_metric": PRIMARY_METRIC,
                "primary_metric_value": float(row[PRIMARY_METRIC]),
                "mean_rank": float(analysis.loc[row["feature"], "mean_rank"]),
                "beats_dummy": bool(
                    analysis.loc[row["feature"], "beats_dummy_rbf"]
                    if model == MODEL_SVC
                    else (
                        analysis.loc[row["feature"], "beats_dummy_rf"]
                        if model == MODEL_RF
                        else False
                    )
                ),
                "selected": bool(analysis.loc[row["feature"], "selected"]),
                "selection_reason": str(analysis.loc[row["feature"], "selection_reason"]),
                "results_validity": RESULTS_VALIDITY,
                "split_fingerprint": split_fingerprint,
            }
        )

for position, feature in enumerate(short_list, start=1):
    row = analysis.loc[feature]
    summary_rows.append(
        {
            "scope": f"combined_short_list_{TOP_K}",
            "model": f"{MODEL_SVC} + {MODEL_RF}",
            "rank_in_model": position,
            "feature": str(feature),
            "primary_metric": PRIMARY_METRIC,
            "primary_metric_value": float(max(row[f"{PRIMARY_METRIC}_rbf_svc"],
                                             row[f"{PRIMARY_METRIC}_random_forest"])),
            "mean_rank": float(row["mean_rank"]),
            "beats_dummy": bool(row["beats_dummy_rbf"] or row["beats_dummy_rf"]),
            "selected": True,
            "selection_reason": str(row["selection_reason"]),
            "results_validity": RESULTS_VALIDITY,
            "split_fingerprint": split_fingerprint,
        }
    )

selection_summary = pd.DataFrame(summary_rows)

output_dir.mkdir(parents=True, exist_ok=True)
selected_path = output_dir / "selected_features.csv"
summary_path = output_dir / "selection_summary.csv"
selected_table.to_csv(selected_path, index=False)
selection_summary.to_csv(summary_path, index=False)

print(f"written: {selected_path.name}  ({selected_table.shape[0]} rows x {selected_table.shape[1]} cols)")
print(f"written: {summary_path.name}  ({selection_summary.shape[0]} rows x {selection_summary.shape[1]} cols)")
print()
print(f"short list ({len(short_list)} features):")
for feature in short_list:
    print(f"  - {feature}")


written: selected_features.csv  (57 rows x 13 cols)
written: selection_summary.csv  (48 rows x 12 cols)

short list (12 features):
  - max_y
  - min_y
  - avg_y
  - min_t
  - max_t
  - avg_t
  - min_x
  - VV2
  - max_x
  - avg_x
  - VV0
  - min_hv


In [26]:
# =============================================================================
#  STEP 9 — WRITE THE STAGE-2 CANDIDATE MANIFEST
#  Records what was read, what was validated, what was chosen and what the
#  limits are. No raw data, no label values, no participant file names.
# =============================================================================
input_hashes = {name: sha256_file(input_dir / name) for name in REQUIRED_INPUT_FILES}

manifest = {
    "results_validity": RESULTS_VALIDITY,
    "notice": (
        "This manifest describes a run of the stage-2 visualisation notebook over the "
        "aggregated outputs of a stage-1 run. It is valid only after a real Colab run on "
        "real stage-1 outputs."
    ),
    "stage": "V3 stage 2 - single-feature results: saving and visualisation",
    "run_timestamp_utc": utc_now_iso(),
    "configuration": {
        "INPUT_DIR": "<withheld: absolute local path is not recorded>",
        "OUTPUT_DIR": "<withheld: absolute local path is not recorded>",
        "PRIMARY_METRIC": PRIMARY_METRIC,
        "TOP_K": TOP_K,
        "RANKING_MODE": RANKING_MODE,
        "FIGURE_DPI": FIGURE_DPI,
        "SAVE_FIGURES": SAVE_FIGURES,
    },
    "environment": {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "matplotlib": matplotlib.__version__,
    },
    "inputs": {
        "files": input_hashes,
        "split_fingerprint": split_fingerprint,
        "cohort_mode": str(metadata.get("cohort_mode", "")),
        "run_status": str(metadata.get("run_status", "")),
        "n_rows_common_cohort": metadata.get("n_rows_common_cohort"),
        "n_train_common": metadata.get("n_train_common"),
        "n_test_common": metadata.get("n_test_common"),
        "n_classes_common": metadata.get("n_classes_common"),
    },
    "validation": {
        "all_checks_passed": all(entry["passed"] for entry in validation_results),
        "n_checks": len(validation_results),
        "checks": validation_results,
    },
    "pool": {
        "n_candidates_screened": int(len(analysis)),
        "n_candidates_in_inventory": n_candidates,
        "n_excluded_columns": n_excluded,
        "n_failure_rows_in_stage1": n_failed_rows,
        "dummy_macro_f1_floor": dummy_shared_value,
        "dummy_floor_identical_across_features": dummy_is_constant,
    },
    "selection": {
        "ranking_mode_description": SUPPORTED_RANKING_MODES[RANKING_MODE],
        "tie_break_order": [
            "the ranking key of RANKING_MODE (mean_rank for the default mode)",
            "the larger of the two models' PRIMARY_METRIC values, descending",
            "the feature name, ascending",
        ],
        "short_list_size": len(short_list),
        "selected_features": [str(f) for f in short_list],
        "selected_features_beating_dummy_in_both_models": [
            str(f)
            for f in short_list
            if bool(analysis.loc[f, "beats_dummy_rbf"]) and bool(analysis.loc[f, "beats_dummy_rf"])
        ],
        "selected_features_not_beating_dummy_in_either_model": [
            str(f)
            for f in short_list
            if not bool(analysis.loc[f, "beats_dummy_rbf"])
            and not bool(analysis.loc[f, "beats_dummy_rf"])
        ],
        "rank_agreement_spearman_rho": spearman_rho,
    },
    "figures": sorted(saved_figures),
    "limitations": [
        "The ranking reuses the stage-1 held-out metrics as a descriptive short list; it is not a re-estimate of generalisation and not a final model selection.",
        "Only one train/test split was used in stage 1, so the ranking inherits that split's uncertainty.",
        "A single-feature score is evidence about information content, not about achievable performance of a feature set.",
        "The candidate pool is the stage-1 complete-case cohort, which is smaller than the merged data; see the stage-1 manifest for the cohort accounting.",
        "Feature-to-feature correlation could not be computed here: the stage-1 outputs contain no row-level values. See correlation_next_step.md.",
        "The DummyClassifier is a reference floor only and is never part of the ranking.",
        "No model was fitted in this notebook and no raw course data was read.",
    ],
    "privacy": [
        "No raw rows, no label values, no participant-linked file names and no local absolute paths are written.",
        "Only aggregate tables, figures and this manifest are produced.",
    ],
}

manifest_path = output_dir / "stage2_candidate_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as handle:
    json.dump(manifest, handle, indent=2, default=str, ensure_ascii=False)

print(f"written: {manifest_path.name}  ({manifest_path.stat().st_size} bytes)")
print(f"  input files hashed : {len(input_hashes)}")
print(f"  validation checks  : {manifest['validation']['n_checks']} (all passed = "
      f"{manifest['validation']['all_checks_passed']})")
print(f"  selected features  : {len(short_list)}")


written: stage2_candidate_manifest.json  (8050 bytes)
  input files hashed : 5
  validation checks  : 19 (all passed = True)
  selected features  : 12


In [27]:
# =============================================================================
#  STEP 10 — WRITE correlation_next_step.md
#  Explains why feature-to-feature correlation cannot be computed from the
#  stage-1 outputs, and what has to happen before feature sets are built.
# =============================================================================
selected_list_block = "\n".join(f"- `{feature}`" for feature in short_list) or "- (none)"

correlation_note = f"""# Feature correlation before the next stage

**Written by:** `V3/notebooks/02_single_feature_visualisation.ipynb`
**Stage:** V3 stage 2 — single-feature results: saving and visualisation
**Validity:** {RESULTS_VALIDITY}

---

## 1. What this notebook could not do, and why

This notebook reads the five aggregate files produced by stage 1. Those files contain

* one row per (feature × model) with the held-out metric values,
* one row per column of the merged data with its type and screening status,
* one row per excluded column with the rule that excluded it,
* one row per failure, and
* a manifest of the cohort, the split and the configuration.

**They contain no per-row feature values.** A correlation coefficient is a statement about
*paired observations* — it needs the value of column A and the value of column B for the same
row. There is no row-level data anywhere in the stage-1 outputs, so a feature-to-feature
Pearson or Spearman correlation **cannot be computed from them**. Producing a
`feature_correlation.csv` from these files would mean inventing numbers, and inventing a
number that looks like evidence is worse than having no number at all.

**No correlation file is therefore written by this notebook.**

## 2. What Figure 4 actually shows

`figures/rank_agreement_svc_vs_random_forest.png` plots the **rank** each feature receives under
the RBF SVC against the rank it receives under the Random Forest, and reports the Spearman
correlation of those two rankings (rho = {spearman_rho:.4f} for this run).

That is **agreement between two models' performance rankings**. It answers "do the two models
order the features the same way?". It does **not** answer "do these two features behave
similarly?". **This is not a correlation between the feature values**, and it must never be
reported as one. Two features can be uncorrelated yet be ranked identically (both informative
for the same reason), and two highly correlated features can be ranked very differently (one of
them is redundant).

## 3. What has to happen before feature sets are built

The correlation step belongs to a **new notebook**, not to this one, and it must:

1. **Load the raw course CSV in Colab only.** Mount the same data folder stage 1 used, read it in
   memory, and never write it, never copy it and never commit it. Only aggregate results leave
   the notebook.
2. **Rebuild exactly the same cohort as stage 1** — the complete-case cohort over the candidate
   columns, with the same `MIN_CLASS_COUNT` filter and the same normalisation of missing and
   infinite values — so that the correlation matrix describes the rows the stage-1 metrics were
   computed on. A correlation computed on a different row set would not be comparable with the
   stage-1 results.
3. **Restrict the computation to the shortlisted columns** listed in `selected_features.csv`:

{selected_list_block}

   The full candidate pool is not needed for the redundancy check, and restricting the
   computation keeps the output small and the review easy.
4. **Compute both coefficients and report them separately**: Pearson for linear association and
   Spearman for monotonic association. Report the number of rows used and the number of rows
   dropped, so the matrix carries its own caveat.
5. **Treat correlation as a redundancy diagnostic, not a selection criterion.** A high
   coefficient between two shortlisted features means the pair carries overlapping information and
   one of them may be dropped to keep the set small. It does not mean that a feature with a low
   coefficient is good, and it does not license dropping a feature on correlation alone.
6. **Keep the output aggregate**: the correlation matrix, the list of near-duplicate pairs and
   the counts. No raw values, no label values, no participant-linked file names.

## 4. Why this is deliberately not in this notebook

This notebook's contract is to visualise and rank **stage-1 aggregates**. Loading the raw data
here would give it a second, undeclared responsibility (reading course data), would put the raw
data on the path of a notebook whose output is a short list, and would blur the boundary that
keeps stage 1 auditable. The correlation question is a real one, and it deserves its own
notebook with its own data-handling rules — not a shortcut inside the visualisation stage.

**This notebook's responsibility ends at the short list.** The correlation step described above
is the immediate prerequisite for the stage-2 feature-set work, and it has to be written and
approved first.
"""

correlation_path = output_dir / "correlation_next_step.md"
correlation_path.write_text(correlation_note, encoding="utf-8")

print(f"written: {correlation_path.name}  ({correlation_path.stat().st_size} bytes)")
print("  (states why no correlation file is produced here, and what the next notebook must do)")


written: correlation_next_step.md  (4648 bytes)
  (states why no correlation file is produced here, and what the next notebook must do)


## Output reference

Everything is written to `OUTPUT_DIR`. All artefacts are aggregate, carry the
`results_validity` stamp `only valid after a real Colab run`, and contain no raw row, no label
value, no participant-linked file name and no local absolute path.

| File | Content |
| --- | --- |
| `figures/topk_<metric>_<model>.png` | Figure 1 — the Top-K single features of each model, one chart per model |
| `figures/scatter_<metric>_svc_vs_random_forest.png` | Figure 2 — RBF SVC against Random Forest, with the y = x reference line |
| `figures/distribution_<metric>_by_model.png` | Figure 3 — the distribution of the metric across the candidate pool, per model |
| `figures/rank_agreement_svc_vs_random_forest.png` | Figure 4 — agreement of the two models' rankings, with the Spearman rho |
| `figures/candidate_pool_audit.png` | Figure 5 — candidate pool composition and the exclusion rules that produced it |
| `selected_features.csv` | Every candidate feature with its ranks, the dummy comparison, the selection flag and the reason |
| `selection_summary.csv` | Per-model Top-K and the combined short list, in one long table |
| `stage2_candidate_manifest.json` | Input hashes, configuration, the validation report, the selected names and the limitations |
| `correlation_next_step.md` | Why feature correlation cannot be computed here, and what the next notebook must do |

### `selected_features.csv`

| Column | Meaning |
| --- | --- |
| `feature` | Candidate feature name |
| `combined_order` | Position in the combined order (1 = best) |
| `rank_rbf_svc` | Rank of the feature under the RBF SVC (1 = best, average rank for ties) |
| `rank_random_forest` | Rank of the feature under the Random Forest |
| `mean_rank` | Mean of the two ranks — the ranking key of the default mode |
| `macro_f1_rbf_svc` / `macro_f1_random_forest` | The stage-1 held-out macro F1 of the two real models |
| `beats_dummy_rbf` / `beats_dummy_rf` | Whether the feature's macro F1 is above the majority-class floor of the dummy classifier, on the same split |
| `selected` | Whether the feature is in the short list |
| `selection_reason` | The explicit reason for the decision, including the ranking mode, the mean rank and the dummy comparison |
| `results_validity` | Validity stamp |
| `split_fingerprint` | The stage-1 split the numbers come from |

### `selection_summary.csv`

Long format with the columns `scope`, `model`, `rank_in_model`, `feature`, `primary_metric`,
`primary_metric_value`, `mean_rank`, `beats_dummy`, `selected`, `selection_reason`,
`results_validity`, `split_fingerprint`. `scope` is `model_top_<TOP_K>` for the three per-model
lists and `combined_short_list_<TOP_K>` for the combined list.

### `stage2_candidate_manifest.json`

The run contract of this stage: the SHA-256 of every input file, the configuration snapshot, the
environment, the full validation report (every check with its result), the pool counts, the dummy
floor, the selection rules and the tie-break order, the selected names, the rank-agreement rho,
the figure names, the limitations and the privacy statement.

---

## Limitations of this stage

1. **The ranking inherits stage 1's single split.** The metrics were produced by one stratified
   80/20 split, so the short list inherits that split's uncertainty. The order of two features
   whose scores are close should be treated as arbitrary.
2. **The short list is descriptive, not a selection.** These are the stage-1 held-out scores
   reused. No re-estimation, no cross-validation and no hyper-parameter search was performed, and
   nothing here establishes which feature set will generalise best.
3. **The candidate pool is the complete-case cohort**, which is smaller than the merged data. The
   cohort accounting is in the stage-1 manifest and was not re-derived here.
4. **A feature that beats the dummy floor still proves very little.** The dummy floor is the
   majority-class share of the test portion; a single feature that clears it marginally carries
   little information, and the margin is not quantified here.
5. **No feature-to-feature correlation was computed**, because the stage-1 outputs contain no
   row-level values. `correlation_next_step.md` explains what the next notebook must do.
6. **Figures describe one run.** They are regenerated from the input each time and are valid only
   for the input whose hash is recorded in the manifest.

## What the supervisor must confirm

- [ ] The five stage-1 inputs are the output of a **completed** stage-1 run in
      `complete_case_all_candidates` mode, and their hashes in the manifest match.
- [ ] The validation report shows every check as PASS.
- [ ] The short list is plausible against the stage-1 tables, and the features that do **not**
      clear the dummy floor are acknowledged (`selection_reason` says so explicitly).
- [ ] The ranking mode and tie-break order are the ones the project wants before the feature-set
      stage is designed.
- [ ] `correlation_next_step.md` is read, and the correlation notebook is written and approved
      **before** any feature set is built.
- [ ] Nothing raw, participant-linked or path-bearing has entered the outputs.

**Stage 2 of the V3 workflow ends here.** The feature-set evaluation notebook is a separate
deliverable and is not created by this file.
